# Stage 7 — Post-Unseal Four-Domain Transportability Discovery and DDO 2.0 Prototype

This is an explicitly **post-unseal retrospective discovery study**. It reuses the existing frozen ResNet50V2 embeddings for EyePACS 2015, DeepDRiD, APTOS 2019, and IDRiD to construct a complete 4×4 directed edge matrix. It does not claim a new prospective blind validation.

The scientific objective is to test whether representation support, discriminative transportability, and calibration/operating-point transportability are separable phenomena. The notebook performs no image download, no image copy, no image decoding, and no backbone inference. New Drive storage is hard-capped at 0.5 GB.


In [1]:
#@title 07-0. Mount Drive, verify Stage 6, and seal the post-unseal discovery protocol
from google.colab import drive
drive.mount("/content/drive")

import hashlib
import importlib.metadata
import json
import os
import platform
import re
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd


print("================ STAGE 7 POST-UNSEAL DISCOVERY PREFLIGHT ================")

PROJECT_ROOT = Path("/content/drive/MyDrive/Cross-Modal_Diagnostic_Observability")
CODE_ROOT = PROJECT_ROOT / "05_Code" / "Retinal_DR"
TEST_ROOT = (
    PROJECT_ROOT / "06_Data_Records" / "Retinal_DR" /
    "Prospective_Retinal_Blind_Test_v0.1"
)
STAGE3_ROOT = TEST_ROOT / "Stage3_Development_Recoverability_Gate_v0.1"
STAGE4_ROOT = TEST_ROOT / "Stage4_Source_Canonicalisation_v0.1"
STAGE5A_ROOT = TEST_ROOT / "Stage5_Source_Recoverability_And_Axis_Freeze_v0.1"
STAGE5B_ROOT = TEST_ROOT / "Stage5B_Label_Free_Target_Scoring_And_Prediction_Freeze_v0.1"
STAGE6_ROOT = TEST_ROOT / "Stage6_One_Time_Target_Unsealing_And_External_Validation_v0.1"
STAGE7_ROOT = (
    TEST_ROOT /
    "Stage7_PostUnseal_FourDomain_Transportability_Discovery_And_DDO2_Prototype_v0.1"
)
PROTOCOL_ROOT = STAGE7_ROOT / "00_Protocol"
EDGE_ROOT = STAGE7_ROOT / "01_Four_Domain_Edge_Matrix"
PROTOTYPE_ROOT = STAGE7_ROOT / "02_DDO2_Prototype"
RESULT_ROOT = STAGE7_ROOT / "03_Results"
AXIS_ROOT = PROTOTYPE_ROOT / "Frozen_PostUnseal_Source_Axes"
for directory in [PROTOCOL_ROOT, EDGE_ROOT, PROTOTYPE_ROOT, RESULT_ROOT, AXIS_ROOT]:
    directory.mkdir(parents=True, exist_ok=True)

NOTEBOOK_PATH = (
    CODE_ROOT /
    "Retinal_DR_Stage7_PostUnseal_FourDomain_Transportability_Discovery_And_DDO2_Prototype_v0.1.ipynb"
)
STAGE6_FINAL_PATH = STAGE6_ROOT / "02_Results" / "Stage6_External_Validation_Complete_v0.1.json"
SOURCE_MANIFEST_PATH = (
    STAGE4_ROOT / "07_Source_Finalisation" / "01_Final_Source_Manifests" /
    "Stage4G_Final_Canonical_Source_Manifest_v0.1.csv"
)
DEVELOPMENT_LABEL_PATHS = {
    "APTOS_2019": TEST_ROOT / "02_Development" / "APTOS_2019_Development_Labels_v0.1.csv",
    "IDRiD": TEST_ROOT / "02_Development" / "IDRiD_Development_Labels_v0.1.csv",
}
EVALUATION_MANIFEST_PATHS = {
    "APTOS_2019": TEST_ROOT / "03_Sealed_Evaluation" / "APTOS_2019_Sealed_Evaluation_Manifest_v0.1.csv",
    "IDRiD": TEST_ROOT / "03_Sealed_Evaluation" / "IDRiD_Sealed_Evaluation_Manifest_v0.1.csv",
}
STAGE3_EMBEDDING_PATHS = {
    target: STAGE3_ROOT / "01_Frozen_Embeddings" /
    f"{target}_Development_ResNet50V2_L2_Embeddings_v0.1.npz"
    for target in ["APTOS_2019", "IDRiD"]
}
STAGE5A_EMBEDDING_PATHS = {
    source: STAGE5A_ROOT / "01_Frozen_Embeddings" /
    f"{source}_Canonical_ResNet50V2_L2_Embeddings_v0.1.npy"
    for source in ["EyePACS_2015", "DeepDRiD"]
}
STAGE5A_ID_PATHS = {
    source: STAGE5A_ROOT / "01_Frozen_Embeddings" /
    f"{source}_Canonical_ResNet50V2_L2_ImageIDs_v0.1.npy"
    for source in ["EyePACS_2015", "DeepDRiD"]
}
STAGE5A_AXIS_PATHS = {
    source: STAGE5A_ROOT / "02_Frozen_Source_Axes" /
    f"{source}_Frozen_Source_Axis_v0.1.npz"
    for source in ["EyePACS_2015", "DeepDRiD"]
}
STAGE5B_EVALUATION_EMBEDDING_PATHS = {
    target: STAGE5B_ROOT / "01_Label_Free_Scores" /
    f"{target}_Canonical_ResNet50V2_L2_Embeddings_v0.1.npy"
    for target in ["APTOS_2019", "IDRiD"]
}
STAGE5B_EVALUATION_ID_PATHS = {
    target: STAGE5B_ROOT / "01_Label_Free_Scores" /
    f"{target}_Canonical_ResNet50V2_L2_ImageIDs_v0.1.npy"
    for target in ["APTOS_2019", "IDRiD"]
}
STAGE6_EVALUATION_PATHS = {
    target: STAGE6_ROOT / "01_Unsealed_Evaluation" /
    f"{target}_Frozen_Score_And_Unsealed_Label_Evaluation_v0.1.csv"
    for target in ["APTOS_2019", "IDRiD"]
}

PROTOCOL_SEAL_PATH = PROTOCOL_ROOT / "Stage7_PostUnseal_Discovery_Protocol_Seal_v0.1.json"
INPUT_COMMITMENT_PATH = PROTOCOL_ROOT / "Stage7_Input_Integrity_Commitment_v0.1.csv"
RUNTIME_STATE_PATH = RESULT_ROOT / "Stage7_Runtime_State_v0.1.json"
FINAL_RECORD_PATH = RESULT_ROOT / "Stage7_FourDomain_Discovery_Complete_v0.1.json"

DATASETS = ["EyePACS_2015", "DeepDRiD", "APTOS_2019", "IDRiD"]
EXPECTED_STAGE6_FINAL_HASH = "cbe17046eb97a191928d93dc0061a33ef52b630377b3af84103fa530568cb326"
USER_REPORTED_FREE_DRIVE_GB = 8.06
MAXIMUM_NEW_STAGE7_BYTES = 512 * 1024 * 1024
RANDOM_SEED = 20260721
N_BOOTSTRAP = 500
FIXED_THRESHOLD = 0.50
AUC_MINIMUM = 0.70
AUC_CI_LOWER_STRICT_MINIMUM = 0.55
CALIBRATION_DEGRADATION_TOLERANCE = 0.05
OPERATING_POINT_BALANCED_ACCURACY_MINIMUM = 0.70
FROZEN_FEATURE_DIMENSION = 2048


def utc_now():
    return datetime.now(timezone.utc).isoformat()


def sha256_file(path, block_size=1024 * 1024):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        while True:
            block = handle.read(block_size)
            if not block:
                break
            digest.update(block)
    return digest.hexdigest()


def sha256_json(payload):
    raw = json.dumps(payload, sort_keys=True, separators=(",", ":"), ensure_ascii=False).encode("utf-8")
    return hashlib.sha256(raw).hexdigest()


def atomic_json(path, payload):
    temporary = Path(str(path) + ".tmp")
    with temporary.open("w", encoding="utf-8") as handle:
        json.dump(payload, handle, indent=2, ensure_ascii=False)
    os.replace(temporary, path)


def normalised_notebook_source_sha256(path):
    with Path(path).open("r", encoding="utf-8") as handle:
        notebook = json.load(handle)
    payload = []
    for cell in notebook.get("cells", []):
        if cell.get("cell_type") not in {"code", "markdown"}:
            continue
        value = cell.get("source", [])
        value = "".join(value) if isinstance(value, list) else str(value)
        payload.append({"cell_type": cell["cell_type"], "source": value.replace("\r\n", "\n")})
    return sha256_json(payload)


required_paths = [
    NOTEBOOK_PATH, STAGE6_FINAL_PATH, SOURCE_MANIFEST_PATH,
    *DEVELOPMENT_LABEL_PATHS.values(), *EVALUATION_MANIFEST_PATHS.values(),
    *STAGE3_EMBEDDING_PATHS.values(), *STAGE5A_EMBEDDING_PATHS.values(),
    *STAGE5A_ID_PATHS.values(), *STAGE5A_AXIS_PATHS.values(),
    *STAGE5B_EVALUATION_EMBEDDING_PATHS.values(),
    *STAGE5B_EVALUATION_ID_PATHS.values(), *STAGE6_EVALUATION_PATHS.values(),
]
for path in required_paths:
    assert path.is_file(), f"Missing required existing asset: {path}"

with STAGE6_FINAL_PATH.open("r", encoding="utf-8") as handle:
    stage6_final = json.load(handle)
stage6_claim = stage6_final["final_record_sha256"]
stage6_without_claim = dict(stage6_final)
stage6_without_claim.pop("final_record_sha256")
assert sha256_json(stage6_without_claim) == stage6_claim
assert stage6_claim == EXPECTED_STAGE6_FINAL_HASH
assert stage6_final["model_refit"] is False and stage6_final["threshold_tuned"] is False

ANALYSIS_SPEC = {
    "scope": "post-unseal retrospective discovery; not a new prospective validation",
    "domains": DATASETS,
    "edges": "complete 4x4 directed matrix; 12 cross-domain and 4 self-validation edges",
    "representation": "existing frozen ImageNet ResNet50V2 2048-dimensional L2 embeddings",
    "axis": (
        "fixed StandardScaler plus class-balanced L2 logistic regression, C=1, "
        "liblinear, development records weighted to one total weight per eye/component"
    ),
    "outcomes": ["ROC_AUC", "AUC_drop", "ECE10", "Brier", "balanced_accuracy_at_0.5"],
    "label_free_components": [
        "5NN_support_fraction", "fraction_beyond_source_q99", "domain_classifier_AUC",
        "RBF_MMD2", "optimal_assignment_cosine_cost", "confidence", "entropy",
        "ATC_estimated_accuracy", "target_to_source_logit_IQR_ratio",
        "source_class_mixture_Wasserstein_residual", "unlabeled_mixture_prevalence",
    ],
    "source_gate": "self-validation AUC >=0.70 and bootstrap CI lower >0.55",
    "discrimination_pass": "target AUC >=0.70 and bootstrap CI lower >0.55",
    "calibration_pass": (
        "target ECE and Brier each no more than 0.05 worse than source validation"
    ),
    "operating_point_pass": "fixed-threshold balanced accuracy >=0.70",
    "inference_boundary": (
        "descriptive Spearman and leave-one-target-out stability only; no general predictor "
        "claim, no p-values, no final DDO2 fit"
    ),
    "storage": (
        "no image download/copy/decode/backbone inference; new Stage7 artefacts hard-capped at 0.5GB"
    ),
}
environment = {
    "python": platform.python_version(),
    "numpy": importlib.metadata.version("numpy"),
    "pandas": importlib.metadata.version("pandas"),
    "scikit_learn": importlib.metadata.version("scikit-learn"),
    "scipy": importlib.metadata.version("scipy"),
    "matplotlib": importlib.metadata.version("matplotlib"),
}
seal_payload = {
    "stage": "Stage7",
    "decision": "SEALED_POST_UNSEAL_FOUR_DOMAIN_DISCOVERY_PROTOCOL",
    "stage6_final_record_sha256": stage6_claim,
    "notebook_source_sha256": normalised_notebook_source_sha256(NOTEBOOK_PATH),
    "analysis_spec": ANALYSIS_SPEC,
    "analysis_spec_sha256": sha256_json(ANALYSIS_SPEC),
    "environment": environment,
    "user_reported_free_drive_gb": USER_REPORTED_FREE_DRIVE_GB,
    "maximum_new_stage7_bytes": MAXIMUM_NEW_STAGE7_BYTES,
    "sealed_utc": utc_now(),
}
seal_payload["seal_sha256"] = sha256_json(seal_payload)
if PROTOCOL_SEAL_PATH.is_file():
    with PROTOCOL_SEAL_PATH.open("r", encoding="utf-8") as handle:
        existing = json.load(handle)
    existing_without_claim = dict(existing)
    existing_claim = existing_without_claim.pop("seal_sha256")
    assert sha256_json(existing_without_claim) == existing_claim
    assert existing["notebook_source_sha256"] == seal_payload["notebook_source_sha256"]
    assert existing["analysis_spec_sha256"] == seal_payload["analysis_spec_sha256"]
    seal_payload = existing
else:
    atomic_json(PROTOCOL_SEAL_PATH, seal_payload)

input_rows = [{
    "path": str(path), "size_bytes": int(path.stat().st_size), "sha256": sha256_file(path)
} for path in required_paths]
input_commitment = pd.DataFrame(input_rows).sort_values("path").reset_index(drop=True)
input_commitment.to_csv(INPUT_COMMITMENT_PATH, index=False)

runtime_state = {
    "stage7_protocol_seal_sha256": seal_payload["seal_sha256"],
    "stage6_final_record_sha256": stage6_claim,
    "post_unseal": True,
    "image_files_read": False,
    "image_files_copied": False,
    "external_data_downloaded": False,
    "new_storage_limit_bytes": MAXIMUM_NEW_STAGE7_BYTES,
    "last_updated_utc": utc_now(),
}
atomic_json(RUNTIME_STATE_PATH, runtime_state)

print("Stage 6 final record verified:", stage6_claim)
print("Stage 7 protocol seal:", seal_payload["seal_sha256"])
print("Existing input bytes referenced (not copied):", int(input_commitment["size_bytes"].sum()))
print("New Stage 7 storage hard limit (GB):", MAXIMUM_NEW_STAGE7_BYTES / 1024**3)
print("Image files read/copied: False/False")
print("External data downloaded: False")


Mounted at /content/drive
================ STAGE 7 POST-UNSEAL DISCOVERY PREFLIGHT ================
Stage 6 final record verified: cbe17046eb97a191928d93dc0061a33ef52b630377b3af84103fa530568cb326
Stage 7 protocol seal: cb45245a1a0f28a3a822f2386b1c5d63edf40582650abb1c3326a2159c98cb2d
Existing input bytes referenced (not copied): 80350839
New Stage 7 storage hard limit (GB): 0.5
Image files read/copied: False/False
External data downloaded: False


In [2]:
#@title 07-1. Load existing embeddings, harmonise record units, and freeze four source axes
from scipy.special import expit, logit
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler


def normalise_eye_side(value):
    if pd.isna(value):
        return ""
    key = re.sub(r"[^a-z0-9]+", "", str(value).lower())
    if key in {"left", "l", "lefteye", "os"}:
        return "left"
    if key in {"right", "r", "righteye", "od"}:
        return "right"
    return ""


def eye_side_from_image_id(image_id):
    text = str(image_id).lower()
    if re.search(r"(?:^|[_\-\s])(?:left|l)(?:$|[_\-\s])", text):
        return "left"
    if re.search(r"(?:^|[_\-\s])(?:right|r)(?:$|[_\-\s])", text):
        return "right"
    return ""


def source_unit_id(row):
    side = normalise_eye_side(row.get("eye_side", "")) or eye_side_from_image_id(row["image_id"])
    if side:
        return f"{row['dataset']}__PATIENT_{row['patient_id']}__EYE_{side}"
    return f"{row['dataset']}__IMAGE_{row['image_id']}"


def align_frame(frame, ids):
    frame = frame.copy()
    frame["image_id"] = frame["image_id"].astype(str)
    assert frame["image_id"].is_unique
    indexed = frame.set_index("image_id", drop=False)
    assert set(ids) == set(indexed.index)
    return indexed.loc[list(ids)].reset_index(drop=True)


source_manifest = pd.read_csv(SOURCE_MANIFEST_PATH, dtype={"image_id": str, "patient_id": str})
domain_assets = {}
for dataset in ["EyePACS_2015", "DeepDRiD"]:
    ids = np.load(STAGE5A_ID_PATHS[dataset], allow_pickle=False).astype(str)
    embeddings = np.load(STAGE5A_EMBEDDING_PATHS[dataset], mmap_mode="r", allow_pickle=False)
    frame = align_frame(source_manifest[source_manifest["dataset"].astype(str).eq(dataset)], ids)
    frame["partition"] = frame["source_partition"].astype(str)
    frame["unit_id"] = frame.apply(source_unit_id, axis=1)
    assert set(frame["partition"]) == {"development", "validation"}
    assert embeddings.shape == (len(frame), FROZEN_FEATURE_DIMENSION)
    domain_assets[dataset] = {"frame": frame, "embedding": embeddings}

for dataset in ["APTOS_2019", "IDRiD"]:
    development = pd.read_csv(DEVELOPMENT_LABEL_PATHS[dataset], dtype={"image_id": str})
    with np.load(STAGE3_EMBEDDING_PATHS[dataset], allow_pickle=True) as loaded:
        development_ids = loaded["image_id"].astype(str)
        development_embeddings = loaded["embedding"].astype(np.float32)
    development = align_frame(development, development_ids)
    development["partition"] = "development"
    development["unit_id"] = (
        development["dataset"].astype(str) + "__COMPONENT_" +
        development["duplicate_component_id"].astype(str)
    )

    evaluation_manifest = pd.read_csv(EVALUATION_MANIFEST_PATHS[dataset], dtype={"image_id": str})
    evaluation_labels = pd.read_csv(STAGE6_EVALUATION_PATHS[dataset], dtype={"image_id": str})[
        ["dataset", "image_id", "original_dr_grade", "moderate_or_worse_dr"]
    ]
    evaluation = evaluation_manifest.merge(
        evaluation_labels, on=["dataset", "image_id"], how="inner", validate="one_to_one"
    )
    evaluation_ids = np.load(STAGE5B_EVALUATION_ID_PATHS[dataset], allow_pickle=False).astype(str)
    evaluation_embeddings = np.load(
        STAGE5B_EVALUATION_EMBEDDING_PATHS[dataset], mmap_mode="r", allow_pickle=False
    )
    evaluation = align_frame(evaluation, evaluation_ids)
    evaluation["partition"] = "validation"
    evaluation["unit_id"] = (
        evaluation["dataset"].astype(str) + "__COMPONENT_" +
        evaluation["duplicate_component_id"].astype(str)
    )
    frame = pd.concat([development, evaluation], ignore_index=True, sort=False)
    embeddings = np.concatenate([
        development_embeddings, np.asarray(evaluation_embeddings, dtype=np.float32)
    ], axis=0)
    assert embeddings.shape == (len(frame), FROZEN_FEATURE_DIMENSION)
    domain_assets[dataset] = {"frame": frame, "embedding": embeddings}

for dataset, asset in domain_assets.items():
    frame = asset["frame"]
    labels = frame["moderate_or_worse_dr"].astype(int)
    assert set(labels.unique()) == {0, 1}
    assert np.isfinite(np.asarray(asset["embedding"])).all()
    assert np.allclose(np.linalg.norm(np.asarray(asset["embedding"]), axis=1), 1.0, atol=1e-5)
    assert not frame.groupby("unit_id")["moderate_or_worse_dr"].nunique().gt(1).any()


def unit_balanced_weights(unit_ids):
    values = pd.Series(unit_ids, dtype=str)
    return 1.0 / values.map(values.value_counts()).to_numpy(dtype=np.float64)


def create_probe():
    return Pipeline([
        ("scaler", StandardScaler()),
        ("classifier", LogisticRegression(
            C=1.0, class_weight="balanced", solver="liblinear", max_iter=5000,
            random_state=RANDOM_SEED,
        )),
    ])


def raw_axis_from_probe(probe):
    scaler = probe.named_steps["scaler"]
    classifier = probe.named_steps["classifier"]
    coefficient_standardised = classifier.coef_[0].astype(np.float64)
    coefficient_raw = coefficient_standardised / scaler.scale_.astype(np.float64)
    intercept_standardised = float(classifier.intercept_[0])
    intercept_raw = float(
        intercept_standardised - np.dot(
            coefficient_standardised,
            scaler.mean_.astype(np.float64) / scaler.scale_.astype(np.float64),
        )
    )
    return {
        "scaler_mean": scaler.mean_.astype(np.float64),
        "scaler_scale": scaler.scale_.astype(np.float64),
        "coefficient_standardised": coefficient_standardised,
        "intercept_standardised": intercept_standardised,
        "coefficient_raw": coefficient_raw,
        "intercept_raw": intercept_raw,
    }


axes = {}
for dataset in DATASETS:
    if dataset in STAGE5A_AXIS_PATHS:
        with np.load(STAGE5A_AXIS_PATHS[dataset], allow_pickle=False) as loaded:
            axes[dataset] = {
                "scaler_mean": loaded["scaler_mean"].astype(np.float64),
                "scaler_scale": loaded["scaler_scale"].astype(np.float64),
                "coefficient_standardised": loaded["coefficient_standardised"].astype(np.float64),
                "intercept_standardised": float(loaded["intercept_standardised"][0]),
                "coefficient_raw": loaded["coefficient_raw"].astype(np.float64),
                "intercept_raw": float(loaded["intercept_raw"][0]),
                "origin": "Stage5A frozen pre-target axis",
            }
    else:
        asset = domain_assets[dataset]
        mask = asset["frame"]["partition"].eq("development").to_numpy()
        frame = asset["frame"].loc[mask].reset_index(drop=True)
        embeddings = np.asarray(asset["embedding"])[mask]
        probe = create_probe()
        probe.fit(
            embeddings,
            frame["moderate_or_worse_dr"].astype(int).to_numpy(),
            classifier__sample_weight=unit_balanced_weights(frame["unit_id"]),
        )
        axis = raw_axis_from_probe(probe)
        axis["origin"] = "Stage7 post-unseal development-only axis"
        axes[dataset] = axis
        axis_path = AXIS_ROOT / f"{dataset}_PostUnseal_Frozen_Development_Axis_v0.1.npz"
        if axis_path.is_file():
            with np.load(axis_path, allow_pickle=False) as existing:
                assert np.allclose(existing["coefficient_raw"], axis["coefficient_raw"], atol=1e-10)
                assert abs(float(existing["intercept_raw"][0]) - axis["intercept_raw"]) < 1e-10
        else:
            np.savez_compressed(
                axis_path,
                scaler_mean=axis["scaler_mean"], scaler_scale=axis["scaler_scale"],
                coefficient_standardised=axis["coefficient_standardised"],
                intercept_standardised=np.asarray([axis["intercept_standardised"]]),
                coefficient_raw=axis["coefficient_raw"],
                intercept_raw=np.asarray([axis["intercept_raw"]]),
            )
        metadata = {
            "dataset": dataset, "origin": axis["origin"],
            "training_partition": "development", "target_validation_used_for_fit": False,
            "C": 1.0, "class_weight": "balanced", "solver": "liblinear",
            "stage7_protocol_seal_sha256": seal_payload["seal_sha256"],
            "axis_sha256": sha256_file(axis_path),
        }
        metadata_path = AXIS_ROOT / f"{dataset}_PostUnseal_Frozen_Development_Axis_Metadata_v0.1.json"
        if metadata_path.is_file():
            assert json.loads(metadata_path.read_text(encoding="utf-8")) == metadata
        else:
            atomic_json(metadata_path, metadata)


def axis_probability(axis, embeddings):
    logits = np.asarray(embeddings, dtype=np.float64) @ axis["coefficient_raw"] + axis["intercept_raw"]
    return expit(logits), logits


def aggregate_units(frame, embeddings, probabilities=None):
    frame = frame.reset_index(drop=True).copy()
    rows, vectors = [], []
    for unit_id, indices in frame.groupby("unit_id", sort=True).indices.items():
        indices = np.asarray(indices, dtype=np.int64)
        labels = frame.iloc[indices]["moderate_or_worse_dr"].astype(int).unique()
        assert len(labels) == 1
        vector = np.asarray(embeddings[indices], dtype=np.float64).mean(axis=0)
        vector /= np.linalg.norm(vector)
        row = {
            "dataset": str(frame.iloc[indices]["dataset"].iloc[0]),
            "unit_id": str(unit_id), "label": int(labels[0]), "images": int(len(indices)),
        }
        if probabilities is not None:
            probability = float(np.mean(np.asarray(probabilities)[indices]))
            row["probability"] = probability
            row["logit"] = float(logit(np.clip(probability, 1e-12, 1 - 1e-12)))
        rows.append(row)
        vectors.append(vector.astype(np.float32))
    return pd.DataFrame(rows), np.stack(vectors)


unit_assets = {}
inventory_rows = []
for dataset, asset in domain_assets.items():
    unit_assets[dataset] = {}
    for partition in ["development", "validation"]:
        mask = asset["frame"]["partition"].eq(partition).to_numpy()
        table, vectors = aggregate_units(
            asset["frame"].loc[mask], np.asarray(asset["embedding"])[mask]
        )
        unit_assets[dataset][partition] = {"table": table, "embedding": vectors, "mask": mask}
        inventory_rows.append({
            "dataset": dataset, "partition": partition,
            "images": int(mask.sum()), "record_units": int(len(table)),
            "negative_units": int(table["label"].eq(0).sum()),
            "positive_units": int(table["label"].eq(1).sum()),
        })
stage7_inventory = pd.DataFrame(inventory_rows)
display(stage7_inventory)
print("Four source axes ready; validation data used for fitting: False")
print("Image files read/copied: False/False")


,dataset,partition,images,record_units,negative_units,positive_units
0,EyePACS_2015,development,3350,3350,2061,1289
1,EyePACS_2015,validation,1118,1118,674,444
2,DeepDRiD,development,1200,600,340,260
3,DeepDRiD,validation,400,200,110,90
4,APTOS_2019,development,2520,2452,1493,959
5,APTOS_2019,validation,1080,1052,641,411
6,IDRiD,development,408,405,151,254
7,IDRiD,validation,102,102,39,63


Four source axes ready; validation data used for fitting: False
Image files read/copied: False/False


In [3]:
#@title 07-2. Build the complete 4×4 edge matrix: label-free observability and observed outcomes
from scipy.optimize import linear_sum_assignment
from scipy.spatial.distance import cdist, pdist
from scipy.stats import wasserstein_distance
from sklearn.decomposition import PCA
from sklearn.metrics import (
    average_precision_score, brier_score_loss, confusion_matrix, log_loss,
    roc_auc_score,
)
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.neighbors import NearestNeighbors


def calibration_error(labels, probabilities, bins=10):
    labels = np.asarray(labels, dtype=int)
    probabilities = np.asarray(probabilities, dtype=float)
    order = np.argsort(probabilities, kind="mergesort")
    bin_ids = np.empty(len(labels), dtype=int)
    bin_ids[order] = np.minimum(
        np.floor(np.arange(len(labels)) * bins / len(labels)).astype(int), bins - 1
    )
    value = 0.0
    for bin_id in range(bins):
        mask = bin_ids == bin_id
        if np.any(mask):
            value += mask.mean() * abs(probabilities[mask].mean() - labels[mask].mean())
    return float(value)


def outcome_metrics(labels, probabilities):
    labels = np.asarray(labels, dtype=int)
    probabilities = np.clip(np.asarray(probabilities, dtype=float), 1e-12, 1 - 1e-12)
    predictions = (probabilities >= FIXED_THRESHOLD).astype(int)
    tn, fp, fn, tp = confusion_matrix(labels, predictions, labels=[0, 1]).ravel()
    sensitivity = tp / (tp + fn)
    specificity = tn / (tn + fp)
    return {
        "auc": float(roc_auc_score(labels, probabilities)),
        "average_precision": float(average_precision_score(labels, probabilities)),
        "brier": float(brier_score_loss(labels, probabilities)),
        "log_loss": float(log_loss(labels, probabilities, labels=[0, 1])),
        "ece10": calibration_error(labels, probabilities),
        "balanced_accuracy": float((sensitivity + specificity) / 2),
        "sensitivity": float(sensitivity), "specificity": float(specificity),
    }


def auc_interval(labels, probabilities, seed):
    labels = np.asarray(labels, dtype=int)
    probabilities = np.asarray(probabilities, dtype=float)
    negative, positive = np.flatnonzero(labels == 0), np.flatnonzero(labels == 1)
    rng = np.random.default_rng(seed)
    values = np.empty(N_BOOTSTRAP, dtype=float)
    for index in range(N_BOOTSTRAP):
        sample = np.concatenate([
            rng.choice(negative, size=len(negative), replace=True),
            rng.choice(positive, size=len(positive), replace=True),
        ])
        values[index] = roc_auc_score(labels[sample], probabilities[sample])
    return [float(np.quantile(values, 0.025)), float(np.quantile(values, 0.975))]


def deterministic_indices(ids, limit, salt):
    ranked = sorted(
        range(len(ids)),
        key=lambda index: hashlib.sha256(f"{salt}|{ids[index]}".encode()).hexdigest(),
    )
    return np.asarray(ranked[:min(limit, len(ranked))], dtype=int)


def support_components(source_embeddings, target_embeddings):
    neighbors = min(6, len(source_embeddings))
    model = NearestNeighbors(n_neighbors=neighbors, metric="cosine").fit(source_embeddings)
    source_distances = model.kneighbors(source_embeddings, return_distance=True)[0]
    source_reference = source_distances[:, 1:].mean(axis=1)
    target_distances = model.kneighbors(target_embeddings, n_neighbors=min(5, len(source_embeddings)), return_distance=True)[0].mean(axis=1)
    q95, q99 = np.quantile(source_reference, [0.95, 0.99])
    return {
        "support_fraction": float(np.mean(target_distances <= q95)),
        "fraction_beyond_source_q99": float(np.mean(target_distances > q99)),
        "source_knn_q95": float(q95), "source_knn_q99": float(q99),
        "target_mean_knn_distance": float(target_distances.mean()),
    }


def geometry_components(source_table, source_embeddings, target_table, target_embeddings, salt):
    limit = min(128, len(source_table), len(target_table))
    source_index = deterministic_indices(source_table["unit_id"].tolist(), limit, salt + "|S")
    target_index = deterministic_indices(target_table["unit_id"].tolist(), limit, salt + "|T")
    source_sample = source_embeddings[source_index].astype(float)
    target_sample = target_embeddings[target_index].astype(float)
    pooled = np.concatenate([source_sample, target_sample], axis=0)
    domain_labels = np.concatenate([np.zeros(limit, dtype=int), np.ones(limit, dtype=int)])
    components = min(32, pooled.shape[0] - 2, pooled.shape[1])
    projected = PCA(n_components=components, svd_solver="randomized", random_state=RANDOM_SEED).fit_transform(pooled)
    classifier = Pipeline([
        ("scale", StandardScaler()),
        ("logistic", LogisticRegression(C=1.0, class_weight="balanced", solver="liblinear", random_state=RANDOM_SEED)),
    ])
    folds = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)
    domain_probability = cross_val_predict(
        classifier, projected, domain_labels, cv=folds, method="predict_proba"
    )[:, 1]
    domain_auc = float(roc_auc_score(domain_labels, domain_probability))

    squared = cdist(source_sample, source_sample, metric="sqeuclidean")
    positive_distances = squared[np.triu_indices_from(squared, k=1)]
    bandwidth = float(np.median(positive_distances[positive_distances > 0]))
    bandwidth = max(bandwidth, 1e-8)
    kss = np.exp(-cdist(source_sample, source_sample, metric="sqeuclidean") / bandwidth).mean()
    ktt = np.exp(-cdist(target_sample, target_sample, metric="sqeuclidean") / bandwidth).mean()
    kst = np.exp(-cdist(source_sample, target_sample, metric="sqeuclidean") / bandwidth).mean()
    mmd2 = float(kss + ktt - 2 * kst)
    cosine_cost = cdist(source_sample, target_sample, metric="cosine")
    row_index, column_index = linear_sum_assignment(cosine_cost)
    ot_cost = float(cosine_cost[row_index, column_index].mean())
    return {"domain_auc": domain_auc, "rbf_mmd2": mmd2, "ot_cosine_cost": ot_cost}


def mixture_components(source_table, target_logits, source_iqr):
    negative = source_table.loc[source_table["label"].eq(0), "logit"].to_numpy(float)
    positive = source_table.loc[source_table["label"].eq(1), "logit"].to_numpy(float)
    target_logits = np.asarray(target_logits, dtype=float)
    values = np.concatenate([negative, positive])
    best = None
    for prevalence in np.linspace(0.0, 1.0, 101):
        weights = np.concatenate([
            np.full(len(negative), (1 - prevalence) / len(negative)),
            np.full(len(positive), prevalence / len(positive)),
        ])
        distance = wasserstein_distance(
            target_logits, values,
            u_weights=np.full(len(target_logits), 1 / len(target_logits)),
            v_weights=weights,
        )
        if best is None or distance < best[0]:
            best = (float(distance), float(prevalence))
    return {
        "mixture_wasserstein_residual_normalised": float(best[0] / max(source_iqr, 1e-8)),
        "unlabeled_mixture_prevalence": best[1],
    }


source_validation = {}
for source_index, source in enumerate(DATASETS):
    asset = domain_assets[source]
    mask = asset["frame"]["partition"].eq("validation").to_numpy()
    probabilities, _ = axis_probability(axes[source], np.asarray(asset["embedding"])[mask])
    table, vectors = aggregate_units(
        asset["frame"].loc[mask], np.asarray(asset["embedding"])[mask], probabilities
    )
    metrics = outcome_metrics(table["label"], table["probability"])
    interval = auc_interval(table["label"], table["probability"], RANDOM_SEED + source_index)
    source_validation[source] = {
        "table": table, "embedding": vectors, "metrics": metrics, "auc_ci": interval,
        "recoverable": bool(metrics["auc"] >= AUC_MINIMUM and interval[0] > AUC_CI_LOWER_STRICT_MINIMUM),
    }

edge_rows, prediction_rows = [], []
for source_index, source in enumerate(DATASETS):
    source_dev_asset = domain_assets[source]
    source_dev_mask = source_dev_asset["frame"]["partition"].eq("development").to_numpy()
    source_dev_probabilities, _ = axis_probability(
        axes[source], np.asarray(source_dev_asset["embedding"])[source_dev_mask]
    )
    source_dev_table, source_dev_vectors = aggregate_units(
        source_dev_asset["frame"].loc[source_dev_mask],
        np.asarray(source_dev_asset["embedding"])[source_dev_mask],
        source_dev_probabilities,
    )
    source_reference = source_validation[source]
    source_iqr = float(np.subtract(*np.quantile(source_reference["table"]["logit"], [0.75, 0.25])))
    source_error = 1 - float(
        np.mean(
            (source_reference["table"]["probability"].to_numpy() >= FIXED_THRESHOLD).astype(int) ==
            source_reference["table"]["label"].to_numpy(int)
        )
    )
    source_confidence = np.maximum(
        source_reference["table"]["probability"].to_numpy(),
        1 - source_reference["table"]["probability"].to_numpy(),
    )
    atc_threshold = float(np.quantile(source_confidence, source_error))

    for target_index, target in enumerate(DATASETS):
        target_asset = domain_assets[target]
        target_mask = target_asset["frame"]["partition"].eq("validation").to_numpy()
        image_probabilities, _ = axis_probability(
            axes[source], np.asarray(target_asset["embedding"])[target_mask]
        )
        target_table, target_vectors = aggregate_units(
            target_asset["frame"].loc[target_mask],
            np.asarray(target_asset["embedding"])[target_mask], image_probabilities,
        )
        outcomes = outcome_metrics(target_table["label"], target_table["probability"])
        interval = auc_interval(
            target_table["label"], target_table["probability"],
            RANDOM_SEED + 100 * source_index + target_index,
        )
        support = support_components(source_dev_vectors, target_vectors)
        geometry = geometry_components(
            source_dev_table, source_dev_vectors, target_table, target_vectors,
            f"{source}|{target}",
        )
        target_iqr = float(np.subtract(*np.quantile(target_table["logit"], [0.75, 0.25])))
        mixture = mixture_components(source_dev_table, target_table["logit"], source_iqr)
        probabilities = target_table["probability"].to_numpy(float)
        confidence = np.maximum(probabilities, 1 - probabilities)
        entropy = -(
            probabilities * np.log(np.clip(probabilities, 1e-12, 1)) +
            (1 - probabilities) * np.log(np.clip(1 - probabilities, 1e-12, 1))
        )
        d_pass = bool(outcomes["auc"] >= AUC_MINIMUM and interval[0] > AUC_CI_LOWER_STRICT_MINIMUM)
        c_pass = bool(
            outcomes["ece10"] <= source_reference["metrics"]["ece10"] + CALIBRATION_DEGRADATION_TOLERANCE and
            outcomes["brier"] <= source_reference["metrics"]["brier"] + CALIBRATION_DEGRADATION_TOLERANCE
        )
        o_pass = bool(outcomes["balanced_accuracy"] >= OPERATING_POINT_BALANCED_ACCURACY_MINIMUM)
        if d_pass and c_pass and o_pass:
            state = "DISCRIMINATION_CALIBRATION_AND_OPERATING_POINT_RETAINED"
        elif d_pass and not c_pass:
            state = "DISCRIMINATION_RETAINED_BUT_CALIBRATION_FAILED"
        elif d_pass and c_pass and not o_pass:
            state = "DISCRIMINATION_AND_CALIBRATION_RETAINED_BUT_OPERATING_POINT_FAILED"
        else:
            state = "DISCRIMINATION_FAILURE_WITH_OR_WITHOUT_ADDITIONAL_FAILURES"

        row = {
            "edge_id": f"{source}__TO__{target}", "source": source, "target": target,
            "is_cross_domain": bool(source != target),
            "source_recoverable": source_reference["recoverable"],
            "source_validation_auc": source_reference["metrics"]["auc"],
            "source_validation_auc_ci_lower": source_reference["auc_ci"][0],
            "target_units": int(len(target_table)),
            "target_auc": outcomes["auc"], "target_auc_ci_lower": interval[0],
            "target_auc_ci_upper": interval[1],
            "source_minus_target_auc": source_reference["metrics"]["auc"] - outcomes["auc"],
            "target_average_precision": outcomes["average_precision"],
            "target_brier": outcomes["brier"], "target_ece10": outcomes["ece10"],
            "calibration_ece_degradation": outcomes["ece10"] - source_reference["metrics"]["ece10"],
            "calibration_brier_degradation": outcomes["brier"] - source_reference["metrics"]["brier"],
            "target_balanced_accuracy_at_0_5": outcomes["balanced_accuracy"],
            "target_sensitivity_at_0_5": outcomes["sensitivity"],
            "target_specificity_at_0_5": outcomes["specificity"],
            "mean_confidence": float(confidence.mean()),
            "mean_entropy_nats": float(entropy.mean()),
            "atc_estimated_accuracy": float(np.mean(confidence >= atc_threshold)),
            "atc_threshold_from_source_validation": atc_threshold,
            "target_to_source_logit_iqr_ratio": float(target_iqr / max(source_iqr, 1e-8)),
            **support, **geometry, **mixture,
            "discrimination_pass": d_pass, "calibration_pass": c_pass,
            "operating_point_pass": o_pass, "observed_transportability_state": state,
        }
        edge_rows.append(row)
        for record in target_table.itertuples(index=False):
            prediction_rows.append({
                "edge_id": row["edge_id"], "source": source, "target": target,
                "unit_id": record.unit_id, "label": int(record.label),
                "probability": float(record.probability), "images": int(record.images),
            })

edge_matrix = pd.DataFrame(edge_rows)
unit_predictions = pd.DataFrame(prediction_rows)
assert len(edge_matrix) == 16
assert int(edge_matrix["is_cross_domain"].sum()) == 12
display(edge_matrix[[
    "source", "target", "source_recoverable", "target_auc", "target_auc_ci_lower",
    "target_ece10", "target_balanced_accuracy_at_0_5", "support_fraction",
    "domain_auc", "mixture_wasserstein_residual_normalised",
    "observed_transportability_state",
]])
print("Complete directed edges:", len(edge_matrix))
print("Cross-domain edges:", int(edge_matrix["is_cross_domain"].sum()))
print("Image files read/copied: False/False")


,source,target,source_recoverable,target_auc,target_auc_ci_lower,target_ece10,target_balanced_accuracy_at_0_5,support_fraction,domain_auc,mixture_wasserstein_residual_normalised,observed_transportability_state
0,EyePACS_2015,EyePACS_2015,False,0.596468,0.563071,0.359121,0.577643,0.942755,0.472229,0.178582,DISCRIMINATION_FAILURE_WITH_OR_WITHOUT_ADDITIO...
1,EyePACS_2015,DeepDRiD,False,0.833434,0.774593,0.130178,0.732323,0.945000,0.991089,0.113779,DISCRIMINATION_CALIBRATION_AND_OPERATING_POINT...
2,EyePACS_2015,APTOS_2019,False,0.838513,0.813885,0.266227,0.739549,0.815589,0.980469,0.501696,DISCRIMINATION_CALIBRATION_AND_OPERATING_POINT...
3,EyePACS_2015,IDRiD,False,0.705332,0.591931,0.302243,0.597070,0.784314,0.991830,0.505485,DISCRIMINATION_AND_CALIBRATION_RETAINED_BUT_OP...
4,DeepDRiD,EyePACS_2015,True,0.634402,0.605592,0.315606,0.591193,0.278175,0.992676,0.303183,DISCRIMINATION_FAILURE_WITH_OR_WITHOUT_ADDITIO...
5,DeepDRiD,DeepDRiD,True,0.924646,0.886301,0.039725,0.830808,0.960000,0.606384,0.267163,DISCRIMINATION_CALIBRATION_AND_OPERATING_POINT...
6,DeepDRiD,APTOS_2019,True,0.675678,0.640576,0.304899,0.621637,0.115970,0.996155,0.321288,DISCRIMINATION_FAILURE_WITH_OR_WITHOUT_ADDITIO...
7,DeepDRiD,IDRiD,True,0.782662,0.690619,0.251944,0.636752,0.000000,1.000000,0.410316,DISCRIMINATION_RETAINED_BUT_CALIBRATION_FAILED
8,APTOS_2019,EyePACS_2015,True,0.655973,0.623408,0.309630,0.582491,0.303220,0.945312,0.083107,DISCRIMINATION_FAILURE_WITH_OR_WITHOUT_ADDITIO...
9,APTOS_2019,DeepDRiD,True,0.884040,0.835290,0.270457,0.678788,0.435000,1.000000,0.142319,DISCRIMINATION_RETAINED_BUT_CALIBRATION_FAILED


Complete directed edges: 16
Cross-domain edges: 12
Image files read/copied: False/False


In [4]:
#@title 07-3. Discover stable relationships, generate the DDO 2.0 candidate map, and freeze outputs
import matplotlib.pyplot as plt
from scipy.stats import spearmanr


LABEL_FREE_COMPONENTS = [
    "support_fraction", "fraction_beyond_source_q99", "target_mean_knn_distance",
    "domain_auc", "rbf_mmd2", "ot_cosine_cost", "mean_confidence",
    "mean_entropy_nats", "atc_estimated_accuracy", "target_to_source_logit_iqr_ratio",
    "mixture_wasserstein_residual_normalised", "unlabeled_mixture_prevalence",
]
OUTCOMES = [
    "target_auc", "source_minus_target_auc", "target_ece10",
    "calibration_ece_degradation", "target_balanced_accuracy_at_0_5",
]


def correlation_rows(frame, scope):
    rows = []
    for component in LABEL_FREE_COMPONENTS:
        for outcome in OUTCOMES:
            subset = frame[[component, outcome]].replace([np.inf, -np.inf], np.nan).dropna()
            rho = float(spearmanr(subset[component], subset[outcome]).statistic) if len(subset) >= 4 else np.nan
            rows.append({
                "scope": scope, "component": component, "outcome": outcome,
                "n_edges": int(len(subset)), "spearman_rho": rho,
            })
    return rows


cross_edges = edge_matrix[edge_matrix["is_cross_domain"]].copy()
gated_edges = cross_edges[cross_edges["source_recoverable"]].copy()
correlation_records = correlation_rows(cross_edges, "all_cross_edges")
correlation_records += correlation_rows(gated_edges, "recoverable_source_cross_edges")
correlation_table = pd.DataFrame(correlation_records)

stability_rows = []
for component in LABEL_FREE_COMPONENTS:
    for outcome in OUTCOMES:
        fold_values = []
        for heldout_target in DATASETS:
            fold = gated_edges[gated_edges["target"].ne(heldout_target)][[component, outcome]].dropna()
            if len(fold) >= 4 and fold[component].nunique() > 1 and fold[outcome].nunique() > 1:
                fold_values.append(float(spearmanr(fold[component], fold[outcome]).statistic))
        nonzero = [value for value in fold_values if np.isfinite(value) and value != 0]
        sign_consistency = (
            max(sum(value > 0 for value in nonzero), sum(value < 0 for value in nonzero)) / len(nonzero)
            if nonzero else np.nan
        )
        stability_rows.append({
            "component": component, "outcome": outcome,
            "valid_leave_one_target_out_folds": int(len(fold_values)),
            "median_spearman_rho": float(np.median(fold_values)) if fold_values else np.nan,
            "minimum_spearman_rho": float(np.min(fold_values)) if fold_values else np.nan,
            "maximum_spearman_rho": float(np.max(fold_values)) if fold_values else np.nan,
            "sign_consistency": float(sign_consistency) if np.isfinite(sign_consistency) else np.nan,
        })
stability_table = pd.DataFrame(stability_rows)

candidate_rows = []
for outcome in OUTCOMES:
    candidates = stability_table[
        stability_table["outcome"].eq(outcome) &
        stability_table["valid_leave_one_target_out_folds"].ge(3) &
        stability_table["sign_consistency"].ge(0.75)
    ].copy()
    if len(candidates):
        candidates["absolute_median_rho"] = candidates["median_spearman_rho"].abs()
        best = candidates.sort_values(
            ["absolute_median_rho", "sign_consistency", "component"],
            ascending=[False, False, True],
        ).iloc[0]
        candidate_rows.append({
            "outcome": outcome, "candidate_component": best["component"],
            "median_leave_one_target_out_rho": best["median_spearman_rho"],
            "sign_consistency": best["sign_consistency"],
            "status": "DISCOVERY_CANDIDATE_REQUIRES_NEW_DATASET_VALIDATION",
        })
    else:
        candidate_rows.append({
            "outcome": outcome, "candidate_component": None,
            "median_leave_one_target_out_rho": None, "sign_consistency": None,
            "status": "NO_STABLE_CANDIDATE_IN_CURRENT_FOUR_DOMAIN_MATRIX",
        })
ddo2_candidates = pd.DataFrame(candidate_rows)

recoverable_sources = sorted(
    source for source in DATASETS if source_validation[source]["recoverable"]
)
stable_primary_candidates = ddo2_candidates[
    ddo2_candidates["outcome"].isin(["target_auc", "target_ece10"]) &
    ddo2_candidates["candidate_component"].notna()
]
if len(recoverable_sources) >= 2 and len(gated_edges) >= 4 and len(stable_primary_candidates) == 2:
    decision = "GO_EXPAND_DATASETS_THEN_FIT_DDO2_USING_DISCOVERED_THREE_AXIS_COMPONENTS"
else:
    decision = "INSUFFICIENT_STABLE_FOUR_DOMAIN_SIGNAL_EXPAND_DATA_BEFORE_DDO2_METHOD_FIT"

display(ddo2_candidates)
state_summary = (
    cross_edges.groupby(["observed_transportability_state"], as_index=False)
    .agg(edges=("edge_id", "size"), sources=("source", "nunique"), targets=("target", "nunique"))
)
display(state_summary)


def write_csv(path, frame):
    text = frame.to_csv(index=False, lineterminator="\n", float_format="%.12g")
    if path.is_file():
        assert path.read_text(encoding="utf-8") == text
    else:
        path.write_text(text, encoding="utf-8")


write_csv(EDGE_ROOT / "Stage7_FourDomain_Edge_Matrix_v0.1.csv", edge_matrix)
write_csv(EDGE_ROOT / "Stage7_FourDomain_Unit_Predictions_v0.1.csv", unit_predictions)
write_csv(EDGE_ROOT / "Stage7_Dataset_Partition_Inventory_v0.1.csv", stage7_inventory)
write_csv(PROTOTYPE_ROOT / "Stage7_LabelFree_Component_Outcome_Correlations_v0.1.csv", correlation_table)
write_csv(PROTOTYPE_ROOT / "Stage7_LeaveOneTargetOut_Component_Stability_v0.1.csv", stability_table)
write_csv(PROTOTYPE_ROOT / "Stage7_DDO2_Discovery_Candidates_v0.1.csv", ddo2_candidates)
write_csv(PROTOTYPE_ROOT / "Stage7_Observed_Transportability_State_Summary_v0.1.csv", state_summary)

figure_path = RESULT_ROOT / "Stage7_ThreeAxis_Transportability_Discovery_v0.1.png"
if not figure_path.is_file():
    fig, axes_plot = plt.subplots(2, 2, figsize=(11, 9))
    panels = [
        ("support_fraction", "target_auc", "Support fraction", "Target AUC"),
        ("support_fraction", "target_ece10", "Support fraction", "Target ECE10"),
        ("domain_auc", "target_auc", "Domain classifier AUC", "Target AUC"),
        ("mixture_wasserstein_residual_normalised", "target_auc", "Mixture residual", "Target AUC"),
    ]
    colours = {"EyePACS_2015": "#868e96", "DeepDRiD": "#1864ab", "APTOS_2019": "#d9480f", "IDRiD": "#2b8a3e"}
    for axis_plot, (x, y, x_label, y_label) in zip(axes_plot.ravel(), panels):
        for source in DATASETS:
            subset = cross_edges[cross_edges["source"].eq(source)]
            axis_plot.scatter(subset[x], subset[y], s=55, alpha=0.85, label=source, color=colours[source])
        axis_plot.set(xlabel=x_label, ylabel=y_label)
        axis_plot.grid(alpha=0.2)
    axes_plot[0, 0].legend(fontsize=8)
    fig.suptitle("Stage 7 post-unseal three-axis transportability discovery")
    fig.tight_layout()
    fig.savefig(figure_path, dpi=180, bbox_inches="tight")
    plt.close(fig)

report_lines = [
    "# Stage 7 — Four-Domain Transportability Discovery", "",
    f"- Decision: `{decision}`",
    f"- Recoverable sources: `{recoverable_sources}`",
    f"- Complete directed edges: `{len(edge_matrix)}`",
    f"- Cross-domain edges: `{len(cross_edges)}`",
    f"- Recoverable-source cross edges: `{len(gated_edges)}`", "",
    "## Boundary", "",
    "This is post-unseal retrospective discovery. Candidate components are not a fitted or externally validated transfer predictor. New datasets are required before any final DDO 2.0 method claim.", "",
    "## Storage", "",
    "No image was downloaded, copied, decoded, or embedded. Stage 7 reused existing arrays and stores only axes, tables, records, and one figure.", "",
]
report_path = RESULT_ROOT / "Stage7_FourDomain_Transportability_Discovery_Report_v0.1.md"
report_text = "\n".join(report_lines)
if report_path.is_file():
    assert report_path.read_text(encoding="utf-8") == report_text
else:
    report_path.write_text(report_text, encoding="utf-8")

output_candidates = sorted([
    path for path in STAGE7_ROOT.rglob("*")
    if path.is_file() and path not in {RUNTIME_STATE_PATH, FINAL_RECORD_PATH}
    and "Output_Integrity_Manifest" not in path.name
], key=str)
output_manifest = pd.DataFrame([{
    "relative_path": str(path.relative_to(STAGE7_ROOT)),
    "size_bytes": int(path.stat().st_size), "sha256": sha256_file(path),
} for path in output_candidates])
output_manifest_path = RESULT_ROOT / "Stage7_Output_Integrity_Manifest_v0.1.csv"
write_csv(output_manifest_path, output_manifest)
new_stage7_bytes = int(
    sum(path.stat().st_size for path in output_candidates) +
    output_manifest_path.stat().st_size
)
assert new_stage7_bytes <= MAXIMUM_NEW_STAGE7_BYTES, (
    f"Stage 7 exceeded its 0.5GB storage cap: {new_stage7_bytes} bytes"
)

final_payload = {
    "stage": "Stage7", "decision": decision,
    "scope": "POST_UNSEAL_RETROSPECTIVE_DISCOVERY_NOT_PROSPECTIVE_VALIDATION",
    "stage6_final_record_sha256": EXPECTED_STAGE6_FINAL_HASH,
    "stage7_protocol_seal_sha256": seal_payload["seal_sha256"],
    "datasets": DATASETS, "directed_edges": int(len(edge_matrix)),
    "cross_domain_edges": int(len(cross_edges)),
    "recoverable_sources": recoverable_sources,
    "recoverable_source_cross_edges": int(len(gated_edges)),
    "observed_state_counts": state_summary.set_index("observed_transportability_state")["edges"].astype(int).to_dict(),
    "ddo2_discovery_candidates": json.loads(
        ddo2_candidates.to_json(orient="records")
    ),
    "image_files_read": False, "image_files_copied": False,
    "external_data_downloaded": False, "backbone_inference_run": False,
    "new_stage7_bytes": new_stage7_bytes,
    "maximum_new_stage7_bytes": MAXIMUM_NEW_STAGE7_BYTES,
    "output_integrity_manifest_sha256": sha256_file(output_manifest_path),
    "next_step": "ADD_INDEPENDENT_DATASETS_BEFORE_FINAL_DDO2_FIT_OR_PROSPECTIVE_VALIDATION",
    "completed_utc": seal_payload["sealed_utc"],
}
final_payload["final_record_sha256"] = sha256_json(final_payload)
if FINAL_RECORD_PATH.is_file():
    with FINAL_RECORD_PATH.open("r", encoding="utf-8") as handle:
        existing = json.load(handle)
    assert existing == final_payload
else:
    atomic_json(FINAL_RECORD_PATH, final_payload)

runtime_state.update({
    "stage7_complete": True, "decision": decision,
    "image_files_read": False, "image_files_copied": False,
    "external_data_downloaded": False, "new_stage7_bytes": new_stage7_bytes,
    "final_record_sha256": final_payload["final_record_sha256"],
    "last_updated_utc": utc_now(),
})
atomic_json(RUNTIME_STATE_PATH, runtime_state)

print("\n================ STAGE 7 FOUR-DOMAIN DISCOVERY COMPLETE ================")
display(edge_matrix[[
    "source", "target", "source_recoverable", "target_auc", "target_auc_ci_lower",
    "target_ece10", "support_fraction", "domain_auc",
    "observed_transportability_state",
]])
display(ddo2_candidates)
print("\nDecision:", decision)
print("Final record:", FINAL_RECORD_PATH)
print("Final record hash:", final_payload["final_record_sha256"])
print("New Stage 7 storage (MB):", new_stage7_bytes / 1024**2)
print("Storage cap (MB):", MAXIMUM_NEW_STAGE7_BYTES / 1024**2)
print("Image files read/copied: False/False")
print("External data downloaded: False")
print("\nSTOP. Interpret the discovery matrix before acquiring new datasets or fitting a final method.")


,outcome,candidate_component,median_leave_one_target_out_rho,sign_consistency,status
0,target_auc,target_mean_knn_distance,-0.517857,1.00,DISCOVERY_CANDIDATE_REQUIRES_NEW_DATASET_VALID...
1,source_minus_target_auc,atc_estimated_accuracy,0.482143,1.00,DISCOVERY_CANDIDATE_REQUIRES_NEW_DATASET_VALID...
2,target_ece10,unlabeled_mixture_prevalence,-0.475234,1.00,DISCOVERY_CANDIDATE_REQUIRES_NEW_DATASET_VALID...
3,calibration_ece_degradation,atc_estimated_accuracy,0.660714,1.00,DISCOVERY_CANDIDATE_REQUIRES_NEW_DATASET_VALID...
4,target_balanced_accuracy_at_0_5,unlabeled_mixture_prevalence,0.465740,0.75,DISCOVERY_CANDIDATE_REQUIRES_NEW_DATASET_VALID...


,observed_transportability_state,edges,sources,targets
0,DISCRIMINATION_AND_CALIBRATION_RETAINED_BUT_OP...,1,1,1
1,DISCRIMINATION_CALIBRATION_AND_OPERATING_POINT...,3,2,2
2,DISCRIMINATION_FAILURE_WITH_OR_WITHOUT_ADDITIO...,4,3,2
3,DISCRIMINATION_RETAINED_BUT_CALIBRATION_FAILED,4,3,3



================ STAGE 7 FOUR-DOMAIN DISCOVERY COMPLETE ================


,source,target,source_recoverable,target_auc,target_auc_ci_lower,target_ece10,support_fraction,domain_auc,observed_transportability_state
0,EyePACS_2015,EyePACS_2015,False,0.596468,0.563071,0.359121,0.942755,0.472229,DISCRIMINATION_FAILURE_WITH_OR_WITHOUT_ADDITIO...
1,EyePACS_2015,DeepDRiD,False,0.833434,0.774593,0.130178,0.945000,0.991089,DISCRIMINATION_CALIBRATION_AND_OPERATING_POINT...
2,EyePACS_2015,APTOS_2019,False,0.838513,0.813885,0.266227,0.815589,0.980469,DISCRIMINATION_CALIBRATION_AND_OPERATING_POINT...
3,EyePACS_2015,IDRiD,False,0.705332,0.591931,0.302243,0.784314,0.991830,DISCRIMINATION_AND_CALIBRATION_RETAINED_BUT_OP...
4,DeepDRiD,EyePACS_2015,True,0.634402,0.605592,0.315606,0.278175,0.992676,DISCRIMINATION_FAILURE_WITH_OR_WITHOUT_ADDITIO...
5,DeepDRiD,DeepDRiD,True,0.924646,0.886301,0.039725,0.960000,0.606384,DISCRIMINATION_CALIBRATION_AND_OPERATING_POINT...
6,DeepDRiD,APTOS_2019,True,0.675678,0.640576,0.304899,0.115970,0.996155,DISCRIMINATION_FAILURE_WITH_OR_WITHOUT_ADDITIO...
7,DeepDRiD,IDRiD,True,0.782662,0.690619,0.251944,0.000000,1.000000,DISCRIMINATION_RETAINED_BUT_CALIBRATION_FAILED
8,APTOS_2019,EyePACS_2015,True,0.655973,0.623408,0.309630,0.303220,0.945312,DISCRIMINATION_FAILURE_WITH_OR_WITHOUT_ADDITIO...
9,APTOS_2019,DeepDRiD,True,0.884040,0.835290,0.270457,0.435000,1.000000,DISCRIMINATION_RETAINED_BUT_CALIBRATION_FAILED


,outcome,candidate_component,median_leave_one_target_out_rho,sign_consistency,status
0,target_auc,target_mean_knn_distance,-0.517857,1.00,DISCOVERY_CANDIDATE_REQUIRES_NEW_DATASET_VALID...
1,source_minus_target_auc,atc_estimated_accuracy,0.482143,1.00,DISCOVERY_CANDIDATE_REQUIRES_NEW_DATASET_VALID...
2,target_ece10,unlabeled_mixture_prevalence,-0.475234,1.00,DISCOVERY_CANDIDATE_REQUIRES_NEW_DATASET_VALID...
3,calibration_ece_degradation,atc_estimated_accuracy,0.660714,1.00,DISCOVERY_CANDIDATE_REQUIRES_NEW_DATASET_VALID...
4,target_balanced_accuracy_at_0_5,unlabeled_mixture_prevalence,0.465740,0.75,DISCOVERY_CANDIDATE_REQUIRES_NEW_DATASET_VALID...



Decision: GO_EXPAND_DATASETS_THEN_FIT_DDO2_USING_DISCOVERED_THREE_AXIS_COMPONENTS
Final record: /content/drive/MyDrive/Cross-Modal_Diagnostic_Observability/06_Data_Records/Retinal_DR/Prospective_Retinal_Blind_Test_v0.1/Stage7_PostUnseal_FourDomain_Transportability_Discovery_And_DDO2_Prototype_v0.1/03_Results/Stage7_FourDomain_Discovery_Complete_v0.1.json
Final record hash: 293db1a6c41f86bcd4c94d91c2d6ad7dfdb5a9369fc4312beacce73b914cd4be
New Stage 7 storage (MB): 1.3643827438354492
Storage cap (MB): 512.0
Image files read/copied: False/False
External data downloaded: False

STOP. Interpret the discovery matrix before acquiring new datasets or fitting a final method.


## Stop boundary

The Stage 7 result is a post-unseal discovery map. Do not call the selected components a validated method. The next legitimate upgrade is dataset acquisition planned around the observed gaps, followed by dataset-level held-out or prospective validation.
